# MA719 Project

## Fetch live data from the internet

I fetch two different kind of datasets:
1. all the german cities with data available -> this is saved with some additional information into `all_cities.json`
2. insolvencies information on all previously fetched cities. This list of json files is saved into `insolvencies.json`

Additionally I implemented some error handling for each fetch.
If nothing could be retrieved the program tries to read an old state from the filesystem.
On success this data is used for the rest of the program, on fail however the program terminates.

In [2]:
import os
import requests
import json
from pathlib import Path

ROOT_URL = "https://infranode.dev/api/v1/cities"
CITIES_FILE_PATH = "all_cities.json"
INSOLVENCIES_FILE_PATH = "insolvencies.json"

def fetch_data_from_url(url: str, _params: str = "") -> json:
    """Tries to fetch data from the given URL"""
    """Returns None on fails"""
    try: 
        response = requests.get(url, params=_params, timeout=10)
        response.raise_for_status()
        print(f"Successfully fetched live data from: {url}")
        return response.json()
    except Exception as e:
        print("Could not reach the service")
        return None

def readJsonFromFile(filepath: str) -> json:
    if (not Path(filepath).exists):
        print("File does not exist")
        return None

    with open(filepath, "r") as f:
        return json.load(f)

def saveJsonToFile(filepath: str, data: json) -> None:
    """Saves json datatype to given file"""
    if not Path(filepath).exists:
        print("File does not exist")
        return 
    
    with(open(filepath, "w")) as f:
        json.dump(data, f, indent=2)


all_cities = fetch_data_from_url(ROOT_URL)
if all_cities == None:
    print("Failed to retrieve cities Data - trying backup file")
    all_cities = readJsonFromFile(CITIES_FILE_PATH)
    if (all_cities == None):
        print("Failed to read all_cites data from backup file")
        exit(1)
else:
    saveJsonToFile(CITIES_FILE_PATH, all_cities)

insolvencies = []
for city in all_cities["data"]:
    slug = city.get("slug")
    url = ROOT_URL+"/"+slug+"/insolvencies"
    data = fetch_data_from_url(url)
    if (data == None):
        print(f"Failed to fetch data from: {url}")
        print("Continuing anyway")
        continue # skip append
        
    insolvencies.append(data)
print(f"Collected data from {len(insolvencies)}/{len(all_cities["data"])} cities")

if insolvencies == None:
    print("Failed to retrieve insolvencies data - trying backup file")
    insolvencies = readJsonFromFile(INSOLVENCIES_FILE_PATH)
    if insolvencies == None:
        print("Failed to read insolvencies from backup file")
        exit(1)
else: 
    saveJsonToFile(INSOLVENCIES_FILE_PATH, insolvencies)


Successfully fetched live data from: https://infranode.dev/api/v1/cities
Successfully fetched live data from: https://infranode.dev/api/v1/cities/berlin/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/hamburg/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/muenchen/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/koeln/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/frankfurt-am-main/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/stuttgart/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/duesseldorf/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/leipzig/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/cities/dortmund/insolvencies
Successfully fetched live data from: https://infranode.dev/api/v1/

## Build pandas data frame 

I create data frames from both json files and merge them together using.

The resulting data frame has all the data I need for my analysis: e.g.: city, federal state, population, amount of insolvencies, year, ...

The data frame contains some unused additional information which I do not discard. The dataset is not that huge so I don't think this is going to be a problem.

In [3]:
import pandas as pd

# read_json only works for flat lists. The provided json files are deeply nested so I had to use a different command
df_all_cities = pd.json_normalize(all_cities["data"])
# df_all_cities.head()

df_insolvencies = pd.json_normalize(insolvencies)
# df_insolvencies.head()

df = pd.merge(df_all_cities, df_insolvencies, left_on="slug", right_on="data.city_slug") # keys on left and right df have different names 
df.head()

#fed_state = df["state"].unique()
#print(fed_state) 

,slug,name_de,state,ags,is_state_capital,qid,osm_relation,population,dwd_station_ids,gtfs_stop_ref,...,data.attribution.license_url,data.attribution.modified,data.payload.kind,data.payload.corporate_insolvencies,data.payload.other_debtor_insolvencies,data.payload.year,data.content_hash,data.record_id,meta.correlation_id,meta.source_status
0,berlin,Berlin,BE,11000000,True,Q64,62422,3782202,[],None,...,https://www.govdata.de/dl-de/by-2-0,False,insolvencies,2092,4551,2024,eecb79b57e0e07df0fb50e6652ae178f18f3dcc0523571...,149ee1065bf8837fefc403e637c870a3e7999af9f8378b...,aac44aea530943a1b66989aa8cd6be04,ok
1,hamburg,Hamburg,HH,02000000,True,Q1055,62782,1910160,[],None,...,https://www.govdata.de/dl-de/by-2-0,False,insolvencies,849,3209,2024,ff56254e0c527d038a66c9af20f74993df8ee0c278fc81...,6551428f1c8fb0a0aa7d6803b3dc02044f3bda8194ba8b...,2be9ad7e89ae4cb4a9c0523585df5073,ok
2,muenchen,München,BY,09162000,True,Q1726,62428,1510378,[],None,...,https://www.govdata.de/dl-de/by-2-0,False,insolvencies,497,1371,2024,a3f1eec734a39019622fc66d5f52a9e65f29dc7fe6f86b...,5e63155d126e788801ec508d37a1a90231d57eb105dcf2...,877a0ca0482a455b9dafedfee49fff4f,ok
3,koeln,Köln,NW,05315000,False,Q365,62578,1087353,[],None,...,https://www.govdata.de/dl-de/by-2-0,False,insolvencies,427,1612,2024,6272672a249cb22d87334bd06ad3055650e8f7708f7fdf...,0a1e3872ecb167e27dd9e35c3453989daeeb4c620039d3...,f16a0af393d444c4bfea553ae9cfe1a9,ok
4,frankfurt-am-main,Frankfurt am Main,HE,06412000,False,Q1794,62400,775790,[],None,...,https://www.govdata.de/dl-de/by-2-0,False,insolvencies,365,1014,2024,3b2d5a1c7c70639c1bc0b659f3122c5a9369bd5c9678c8...,ba4c30f7c8cbed9c583ca3b4102f628c906a4bef572a60...,af42fe1cc14c4c6d80354e4eb8bb808d,ok


## Data analysis -> reducing DataFrame to relevant data 

I take my pandas data frame and calculate the insolvencies / 100.000 population for each city and safe that value into a new data frame.

This data frame holds: slug, name_de, state and the calculated value --> good usability for further analysis.

In [ ]:
def normalize_data(population : int, insolvencies : int) -> int:
    """Returns the insolvencies per 100.000 population"""
    return (insolvencies / population * 100_000)


In [17]:
# copy only needed columns to new data frame
# src: https://stackoverflow.com/questions/34682828/extracting-specific-selected-columns-to-new-dataframe-as-a-copy
min_data = df[["slug", "name_de", "state", "population", "data.payload.corporate_insolvencies", "data.payload.other_debtor_insolvencies"]].copy()
min_data = min_data.rename(columns={"data.payload.corporate_insolvencies": "corporate_insolvencies", "data.payload.other_debtor_insolvencies": "other_debtor_insolvencies"})

min_data["normalized_insolvencies"] = normalize_data(
    min_data["population"], min_data["corporate_insolvencies"] + min_data["other_debtor_insolvencies"]
)

min_data.head()

,slug,name_de,state,population,corporate_insolvencies,other_debtor_insolvencies,normalized_insolvencies
0,berlin,Berlin,BE,3782202,2092,4551,175.638424
1,hamburg,Hamburg,HH,1910160,849,3209,212.442937
2,muenchen,München,BY,1510378,497,1371,123.677649
3,koeln,Köln,NW,1087353,427,1612,187.519600
4,frankfurt-am-main,Frankfurt am Main,HE,775790,365,1014,177.754289


## Data analysis -> Calculating average for each federal state

Take the data of 84 German cities and calculate the average normalized insolvencies for each federal state.

In [18]:
fed_normalized = min_data[["state", "normalized_insolvencies"]].groupby("state").mean()
fed_normalized.head()

,normalized_insolvencies
state,
BB,192.944777
BE,175.638424
BW,174.540225
BY,130.492548
HB,283.927264


## Data analysis -> difference between west and east

Calculate the average of all west / east german insolvencies. 

I add a `west` / `east` column and groupby.mean on this newly created column.

In [ ]:
def get_east_west(state: str) -> str:
    """Maps German federal states to east / west german region -> returns string containing `west` or `east`"""
    # using official 
    east_germany = [""]